# IEAP – Python – Pre-series 04

**Group members (active):**
- BINCHE Werner
- LAGARDE Alexis

**GitHub repository:** https://github.com/AlexisLag-student/IEAP-Pre-Series04-Python

**Work split:**
- LAGARDE Alexis — Part 1 (generate a known signal) and Part 2 (time derivative)
- BINCHE Werner — Part 3 (tangential speed + tests) and Part 4 (interactive plotly plot)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Generate a known signal

### 1.1 Equation of a sinusoid at 1 Hz, amplitude 1, phase 0

The general sinusoid is $s(t) = A \sin(2\pi f t + \varphi)$. With $A = 1$, $f = 1\ \text{Hz}$ and $\varphi = 0$:

$$
s(t) = \sin(2\pi t)
$$

### 1.2 Time vector

The statement asks for "1000 samples, 100 Hz, 3 seconds", but these three values are not mutually consistent
(at $f_s = 100$ Hz, 3 s contain 300 sampling intervals, not 1000 samples).
We keep the two physically meaningful parameters - **sampling frequency $f_s = 100$ Hz** and **duration 3 s** -
and include both end points, which gives $3 \times 100 + 1 = 301$ samples, with $\Delta t = 1/f_s = 0.01$ s.
The time vector is reshaped as a **column vector** (shape `(n, 1)`), so that samples are over rows.

### 1.3 3D signal with a phase shift between axes

Each column is the same 1 Hz, unit-amplitude sinusoid, shifted by $\Delta\varphi$ from one axis to the next:

$$
x(t) = \sin(2\pi t), \qquad
y(t) = \sin(2\pi t + \Delta\varphi), \qquad
z(t) = \sin(2\pi t + 2\Delta\varphi)
$$


In [ ]:
# parameters
fs = 100            # sampling frequency (Hz)
duration = 3.0      # duration (s)
f = 1.0             # signal frequency (Hz)
A = 1.0             # amplitude
phase_shift = 0.25 * np.pi


n_samples = int(duration * fs) + 1               
time = np.linspace(0, duration, n_samples).reshape(-1, 1)

phases = np.array([[0, 1, 2]]) * phase_shift     
signal = A * np.sin(2 * np.pi * f * time + phases) 

In [ ]:
t, s = time, signal
print(f"t.shape = {t.shape}")
print(f"s.shape = {s.shape}")
print(f"phase_shift = {phase_shift / np.pi:g} pi")

# 2D 
fig, ax = plt.subplots(figsize=(8, 6))
for i, label in enumerate(["x", "y", "z"]):
    ax.plot(t[:, 0], s[:, i], "-.", marker=".", label=label)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.grid(True)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

# 3D 
fig = plt.figure(figsize=(6, 6))
ax3 = fig.add_subplot(projection="3d")
step = 4   # one point out of 4, to keep the trajectory readable
ax3.scatter(s[::step, 0], s[::step, 1], s[::step, 2], label="signal")
ax3.set_xlabel("x"); ax3.set_ylabel("y"); ax3.set_zlabel("z")
ax3.set_title("3D plot of the signal")
ax3.legend()
plt.show()

## 2. Estimate the time derivative of the signal

### 2.1 Derivative of a sine wave

$$
\frac{d}{dt}\big[A\sin(2\pi f t + \varphi)\big] = 2\pi f A \cos(2\pi f t + \varphi)
$$

### 2.2 Time derivative of `signal` along each axis

With $A = 1$, $f = 1$ Hz and $\Delta\varphi = \pi/4$:

$$
\dot{x}(t) = 2\pi\cos(2\pi t), \qquad
\dot{y}(t) = 2\pi\cos\!\left(2\pi t + \tfrac{\pi}{4}\right), \qquad
\dot{z}(t) = 2\pi\cos\!\left(2\pi t + \tfrac{\pi}{2}\right)
$$

Each derivative oscillates between $-2\pi$ and $+2\pi \approx 6.28$.

### 2.3 Forward difference vs. central difference

| | Forward difference | Central difference |
|---|---|---|
| Formula | $\dot{s}_i \approx \dfrac{s_{i+1} - s_i}{t_{i+1} - t_i}$ | $\dot{s}_i \approx \dfrac{s_{i+1} - s_{i-1}}{t_{i+1} - t_{i-1}}$ |
| Accuracy | first order, error $O(h)$ | second order, error $O(h^2)$ |
| Time shift | half a sample (slope at $t_i + h/2$) | none (symmetric around $t_i$) |
| Missing values | last sample | first and last samples |

Missing edge values are filled with a one-sided difference.

**Reference:** https://en.wikipedia.org/wiki/Finite_difference

### 2.4 Design of the functions

- **Input order `(signal, time)`:** we differentiate the signal with respect to time ($d\,\text{signal}/d\,\text{time}$), the same order as `numpy.gradient`. Passing `time` instead of `fs` also handles irregular sampling.
- **Output shape = shape of `signal`** `(n_samples, n_dims)`: one derivative per sample and per axis, so it can be plotted against `time` directly.

In [ ]:
def _as_2d(signal, time):
    # Helper: force signal to (n, d) and time to (n, 1), and check consistency.
    signal = np.asarray(signal, dtype=float)
    time = np.asarray(time, dtype=float)
    if signal.ndim == 1:
        signal = signal.reshape(-1, 1)
    time = time.reshape(-1, 1)
    if signal.shape[0] != time.shape[0]:
        raise ValueError("signal and time must have the same number of samples (rows)")
    if signal.shape[0] < 2:
        raise ValueError("at least 2 samples are needed to compute a derivative")
    return signal, time


def derivative_forward(signal, time):
    # Time derivative with the forward difference (first order, O(h)).
    #   ds/dt[i] = (s[i+1] - s[i]) / (t[i+1] - t[i])     for i = 0 .. n-2
    #   last sample: backward difference (s[n-1] - s[n-2]) / (t[n-1] - t[n-2])
    # Inputs : signal (n, d) samples over rows, dims over columns ; time (n,) or (n, 1)
    # Returns: numpy array of shape (n, d), same shape as signal.
    signal, time = _as_2d(signal, time)
    d = np.empty_like(signal)
    d[:-1] = (signal[1:] - signal[:-1]) / (time[1:] - time[:-1])
    d[-1] = d[-2]          # backward difference at the last sample (same quotient)
    return d

def derivative_central(signal, time):
    # Time derivative with the central difference (second order, O(h^2)).
    #   ds/dt[i] = (s[i+1] - s[i-1]) / (t[i+1] - t[i-1])  for i = 1 .. n-2
    #   first sample: forward difference ; last sample: backward difference
    # Inputs : signal (n, d) samples over rows, dims over columns ; time (n,) or (n, 1)
    # Returns: numpy array of shape (n, d), same shape as signal.
    signal, time = _as_2d(signal, time)
    d = np.empty_like(signal)
    if signal.shape[0] > 2:
        d[1:-1] = (signal[2:] - signal[:-2]) / (time[2:] - time[:-2])
    d[0] = (signal[1] - signal[0]) / (time[1] - time[0])
    d[-1] = (signal[-1] - signal[-2]) / (time[-1] - time[-2])
    return d

In [ ]:
d_fwd = derivative_forward(signal, time)
d_cen = derivative_central(signal, time)
print(f"signal.shape = {signal.shape} -> d_fwd.shape = {d_fwd.shape}, d_cen.shape = {d_cen.shape}")

# Analytical derivative, to quantify the error of each method
d_true = 2 * np.pi * f * A * np.cos(2 * np.pi * f * time + phases)
print(f"max |error| forward difference : {np.max(np.abs(d_fwd - d_true)):.4f}")
print(f"max |error| central difference : {np.max(np.abs(d_cen[1:-1] - d_true[1:-1])):.6f}  (interior samples)")
print(f"np.gradient agrees with derivative_central: {np.allclose(d_cen, np.gradient(signal, time[:, 0], axis=0))}")

for deriv, name in [(d_fwd, "Forward Difference"), (d_cen, "Central Difference")]:
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, ax_name in enumerate(["x", "y", "z"]):
        ax.plot(time, signal[:, i], label=f"Signal {ax_name}")
    for i, ax_name in enumerate(["x", "y", "z"]):
        ax.plot(time, deriv[:, i], label=f"Derivative {ax_name} ({name})")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.grid(True)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
    plt.tight_layout()
    plt.show()

**Cells for Parts 3 and 4 (BINCHE Werner).** They use `signal`, `time`, `phases` and `derivative_central` defined in Parts 1–2, so they only run inside the shared notebook.

## 3. Estimate the tangential speed of the signal

### 3.1 Tangential velocity vs. tangential speed

For a point whose position is $\vec{r}(t)$:

- The **(tangential) velocity** is a **vector**: $\vec{v}(t) = \dfrac{d\vec{r}}{dt}$. It has a magnitude *and* a direction,
  and this direction is always tangent to the trajectory (hence "tangential"). In 3D it has 3 components $(\dot x, \dot y, \dot z)$.
- The **(tangential) speed** is a **scalar**: the magnitude (Euclidean norm) of the velocity vector,
  $v(t) = \lVert\vec{v}(t)\rVert \ge 0$. It is the rate at which the distance travelled along the path (arc length $s$) increases: $v = ds/dt$.
  It has no direction and is never negative.

Example: in uniform circular motion the speed is constant, but the velocity changes all the time (its direction rotates).

**References:** Wikipedia, *Velocity* — https://en.wikipedia.org/wiki/Velocity ;
Wikipedia, *Speed* — https://en.wikipedia.org/wiki/Speed ;
Wikipedia, *Arc length* — https://en.wikipedia.org/wiki/Arc_length

### 3.2 Tangential speed of a 3D movement

$$
v(t) = \lVert\vec{v}(t)\rVert = \sqrt{\dot{x}(t)^2 + \dot{y}(t)^2 + \dot{z}(t)^2}
$$

### 3.3 Tangential speed of an nD movement

For a position $\vec{r}(t) = \big(r_1(t), \dots, r_n(t)\big) \in \mathbb{R}^n$:

$$
v(t) = \lVert\dot{\vec{r}}(t)\rVert_2 = \sqrt{\sum_{k=1}^{n} \dot{r}_k(t)^2}
$$

In 1D this reduces to $v = |\dot{x}|$. For our signal:
$v(t) = 2\pi\sqrt{\cos^2(2\pi t) + \cos^2(2\pi t + \frac{\pi}{4}) + \cos^2(2\pi t + \frac{\pi}{2})}$, which is not constant
(the trajectory is an ellipse, not a circle traversed uniformly).

### 3.4 Function design

- **Algorithm:** (1) estimate the velocity vector with the central difference (more accurate, no time shift),
  (2) take the Euclidean norm of each row (`np.linalg.norm(..., axis=1)`).
- **Input order: `(signal, time)`**, for the same reasons as the derivative functions (quantity first, variable of
  differentiation second; consistent with `derivative_central(signal, time)`, which it calls).
- **Output shape:** `(n_samples, 1)` — one scalar speed per sample, kept as a column vector, consistent with `time`
  so it can be plotted directly against it. It works for any number of dimensions `d` (1D, 2D, 3D, … nD).

In [ ]:
# References: https://en.wikipedia.org/wiki/Speed , https://en.wikipedia.org/wiki/Velocity


def tangential_speed(signal, time):
    # Tangential speed = Euclidean norm of the velocity vector, for an nD movement.
    #   v(t) = sqrt( sum_k (dr_k/dt)^2 )
    # Inputs : signal (n, d) samples over rows, dims over columns (d = 1, 2, 3, ...) ;
    #          time (n,) or (n, 1)
    # Returns: numpy array of shape (n, 1), one speed value per sample.
    velocity = derivative_central(signal, time)          # (n, d)
    return np.linalg.norm(velocity, axis=1, keepdims=True)  # (n, 1)


speed = tangential_speed(signal, time)
print(f"signal.shape = {signal.shape} -> speed.shape = {speed.shape}")
print(f"speed: min = {speed.min():.3f}, max = {speed.max():.3f}")

### 3.5 Test cell

For each test case the expected speed is first derived analytically, then compared to the function output with `assert`.
A tolerance is needed because finite differences are an approximation (relative error of the order of $(\omega h)^2$ here).

| Case | Movement $\vec r(t)$ | Velocity $\dot{\vec r}(t)$ | Expected speed $v(t)$ |
|---|---|---|---|
| 1D linear | $x = a t + b$ | $\dot x = a$ | $\lvert a\rvert$ (constant) |
| 1D sine | $x = A\sin(\omega t)$ | $\dot x = A\omega\cos(\omega t)$ | $A\omega\,\lvert\cos(\omega t)\rvert$ |
| 2D circle | $(R\cos\omega t,\ R\sin\omega t)$ | $(-R\omega\sin\omega t,\ R\omega\cos\omega t)$ | $R\omega$ (constant) |
| 3D helix | $(R\cos\omega t,\ R\sin\omega t,\ c\,t)$ | $(-R\omega\sin\omega t,\ R\omega\cos\omega t,\ c)$ | $\sqrt{R^2\omega^2 + c^2}$ |
| 3D our signal | $\sin(2\pi t + k\Delta\varphi)$, $k=0,1,2$ | $2\pi\cos(2\pi t + k\Delta\varphi)$ | $2\pi\sqrt{\sum_k \cos^2(2\pi t + k\Delta\varphi)}$ |
| nD linear | $\vec r = \vec a\, t + \vec b$, $\vec a\in\mathbb{R}^n$ | $\vec a$ | $\lVert\vec a\rVert$ (e.g. $\vec a = (1,\dots,1)\Rightarrow\sqrt{n}$) |

For linear movements finite differences are **exact** (the second derivative is zero), so the tolerance can be very tight.
For the 1D sine, $\lvert\cos\rvert$ has kinks at its zeros, so the comparison uses an absolute tolerance.

In [ ]:
fs_test = 100
t_test = np.arange(0, 2 + 1 / fs_test, 1 / fs_test).reshape(-1, 1)   # (201, 1)
n = t_test.shape[0]

# --- 1D linear: x = a t + b  ->  v = |a| -----------------------------------------------
a, b = -3.0, 2.0
v = tangential_speed(a * t_test + b, t_test)
assert v.shape == (n, 1)
assert np.allclose(v, abs(a)), "1D linear failed"

# --- 1D sine: x = A sin(wt)  ->  v = A w |cos(wt)| ------------------------------------
A_, w = 2.0, 2 * np.pi * 1.0
v = tangential_speed(A_ * np.sin(w * t_test), t_test)
v_exp = A_ * w * np.abs(np.cos(w * t_test))
assert v.shape == (n, 1)
assert np.allclose(v[1:-1], v_exp[1:-1], atol=0.05), "1D sine failed"   # interior samples

# --- 1D signal given as a flat array (n,) instead of (n, 1) --------------------------
v = tangential_speed((a * t_test + b).ravel(), t_test.ravel())
assert v.shape == (n, 1) and np.allclose(v, abs(a)), "1D flat input failed"

# --- 2D circle: v = R w (constant) ----------------------------------------------------
R = 0.5
circle = np.hstack([R * np.cos(w * t_test), R * np.sin(w * t_test)])   # (n, 2)
v = tangential_speed(circle, t_test)
assert v.shape == (n, 1)
assert np.allclose(v, R * w, rtol=1e-3), "2D circle failed"

# --- 3D helix: v = sqrt(R^2 w^2 + c^2) -----------------------------------------------
c = 0.8
helix = np.hstack([R * np.cos(w * t_test), R * np.sin(w * t_test), c * t_test])   # (n, 3)
v = tangential_speed(helix, t_test)
assert v.shape == (n, 1)
assert np.allclose(v, np.sqrt(R**2 * w**2 + c**2), rtol=1e-3), "3D helix failed"

# --- 3D: the signal of part 1 ------------------------------------------------------
v_exp = 2 * np.pi * np.sqrt(np.sum(np.cos(2 * np.pi * time + phases) ** 2, axis=1, keepdims=True))
v = tangential_speed(signal, time)
assert v.shape == (signal.shape[0], 1)
assert np.allclose(v[1:-1], v_exp[1:-1], rtol=1e-3), "3D signal failed"

# --- nD linear (4D, 5D, ... 10D): a = (1, ..., 1) -> v = sqrt(n_dims) -----------------
for n_dims in range(1, 11):
    a_vec = np.ones((1, n_dims))
    v = tangential_speed(t_test * a_vec + 1.0, t_test)
    assert v.shape == (n, 1)
    assert np.allclose(v, np.sqrt(n_dims)), f"{n_dims}D linear failed"

# --- 4D 'double circle' (Clifford torus): v = sqrt(R1^2 w1^2 + R2^2 w2^2) ----------------
R1, w1, R2, w2 = 1.0, 2 * np.pi, 0.5, 4 * np.pi
torus = np.hstack([R1 * np.cos(w1 * t_test), R1 * np.sin(w1 * t_test),
                   R2 * np.cos(w2 * t_test), R2 * np.sin(w2 * t_test)])   # (n, 4)
v = tangential_speed(torus, t_test)
assert np.allclose(v, np.sqrt(R1**2 * w1**2 + R2**2 * w2**2), rtol=1e-2), "4D torus failed"

print("All tests passed: 1D, 2D, 3D, 4D and nD (up to 10D).")

## 4. Interactive plot with the plotly library

1. Installation (run once in a terminal or a notebook cell): `pip install plotly`
2. The script below plots the three coordinates of the signal and its tangential speed on the same time axis.
3. The plot can be zoomed (drag a box), panned, reset (double-click), and individual traces can be hidden by clicking on the legend.
4. The figure is embedded with its full JavaScript library, so it stays **interactive and zoomable in the exported HTML notebook**,
   even offline. It is also saved as a standalone file `signal_tangential_speed.html` to share with co-authors.

In [ ]:
# %pip install plotly      # uncomment to install from the notebook
import plotly.graph_objects as go
from IPython.display import HTML, display

fig = go.Figure()
for i, ax_name in enumerate(["x", "y", "z"]):
    fig.add_trace(go.Scatter(x=time[:, 0], y=signal[:, i], mode="lines", name=ax_name))
fig.add_trace(go.Scatter(x=time[:, 0], y=speed[:, 0], mode="lines", name="tangential speed",
                         line=dict(width=3)))
fig.update_layout(
    title="Signal (x, y, z) and its tangential speed",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude (m) / Speed (m/s)",
    hovermode="x unified",
    template="plotly_white",
    height=500,
)

# Standalone HTML file (zoomable, works offline)
fig.write_html("signal_tangential_speed.html", include_plotlyjs=True)

# Embed in the notebook so it remains interactive in the HTML export
display(HTML(fig.to_html(full_html=False, include_plotlyjs=True)))

In [ ]:
import nbformat
from nbconvert import HTMLExporter

notebook_file = "IEAP-Python-Series04-pre.ipynb"   # name of this notebook
html_file = notebook_file.replace(".ipynb", ".html")

nb_saved = nbformat.read(notebook_file, as_version=4)
html_body, _ = HTMLExporter().from_notebook_node(nb_saved)
with open(html_file, "w", encoding="utf-8") as fh:
    fh.write(html_body)

print(f"Exported: {html_file}")